In [8]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print('Authenticated')

Authenticated


In [9]:
!pip install google-cloud

### 📌***1. Створення ДатаСету***

✅ ***Підключення до DB bigquery***

In [10]:
PROJECT_ID = "data-analytics-mate"

client = bigquery.Client(project=PROJECT_ID)

✅ ***Запит до DB для формування датасету на основі таблиць***

In [11]:
query = """
    SELECT
    s.date as date,
    s.ga_session_id as session_id,
    sp.continent,
    sp.country,
    sp.device,
    sp.browser,
    sp.mobile_model_name,
    sp.operating_system,
    sp.language,
    sp.medium as traffic_source,
    sp.channel,
    acs.account_id,
    ac.is_verified,
    ac.is_unsubscribed,
    p.category,
    p.name as product_name,
    p.price,
    p.short_description as description
    FROM `DA.session` as s
    LEFT JOIN `DA.session_params` as sp
    ON s.ga_session_id = sp.ga_session_id
    LEFT JOIN `DA.account_session` as acs
    ON s.ga_session_id = acs.ga_session_id
    LEFT JOIN `DA.account` as ac
    ON acs.account_id = ac.id
    LEFT JOIN `DA.order` as o
    ON s.ga_session_id = o.ga_session_id
    LEFT JOIN `DA.product` as p
    ON o.item_id = p.item_id
"""

df = client.query(query).to_dataframe()
df.head()

,date,session_id,continent,country,device,browser,mobile_model_name,operating_system,language,traffic_source,channel,account_id,is_verified,is_unsubscribed,category,product_name,price,description
0,2020-11-01,5760483956,Americas,United States,desktop,Chrome,Safari,Macintosh,zh,<Other>,Paid Search,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
1,2020-11-01,7115337200,Europe,United Kingdom,desktop,Chrome,Chrome,Web,en-us,organic,Organic Search,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
2,2020-11-01,3978035233,Europe,Norway,mobile,Chrome,<Other>,Web,zh,(none),Direct,<NA>,<NA>,<NA>,Tables & desks,RÅSKOG,189.0,"Trolley, 35x45x78 cm"
3,2020-11-01,9648986282,Africa,Nigeria,mobile,Chrome,<Other>,Android,es-es,(none),Direct,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
4,2020-11-01,4393441533,Asia,China,desktop,Chrome,Chrome,Windows,en-us,(none),Direct,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"


###📌***2. Розуміння даних та їх змісту***

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 349545 entries, 0 to 349544
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   date               349545 non-null  dbdate 
 1   session_id         349545 non-null  Int64  
 2   continent          349545 non-null  object 
 3   country            349545 non-null  object 
 4   device             349545 non-null  object 
 5   browser            349545 non-null  object 
 6   mobile_model_name  349545 non-null  object 
 7   operating_system   349545 non-null  object 
 8   language           235279 non-null  object 
 9   traffic_source     349545 non-null  object 
 10  channel            349545 non-null  object 
 11  account_id         27945 non-null   Int64  
 12  is_verified        27945 non-null   Int64  
 13  is_unsubscribed    27945 non-null   Int64  
 14  category           33538 non-null   object 
 15  product_name       33538 non-null   object 
 16  pr

In [13]:
df.describe()

,session_id,account_id,is_verified,is_unsubscribed,price
count,349545.0,27945.0,27945.0,27945.0,33538.000000
mean,4992250296.631739,659005.065557,0.71698,0.16944,953.298679
std,2887450949.537772,13216.529465,0.450474,0.375147,1317.001775
min,1205.0,636133.0,0.0,0.0,3.000000
25%,2493646855.0,647576.0,0.0,0.0,170.000000
50%,4988476074.0,658952.0,1.0,0.0,445.000000
75%,7491286508.0,670414.0,1.0,0.0,1195.000000
max,9999997129.0,681962.0,1.0,1.0,9585.000000


✅ ***Розмір датасету***

In [14]:
df.shape

(349545, 18)

✅ ***Загальна кількість колонок***

In [15]:
print('загальна кількість колонок:', df.shape[1])

загальна кількість колонок: 18


✅ ***Кількість та назва колонок числового типу***

In [16]:
numeric_df = df.select_dtypes(include=['number'])
print('кількість колонок числового типу:',  numeric_df.shape[1])
print('які саме колонки числового типу:',  numeric_df.columns.to_list())

кількість колонок числового типу: 5
які саме колонки числового типу: ['session_id', 'account_id', 'is_verified', 'is_unsubscribed', 'price']


✅ ***Кількість та назва колонок категоріального типу***

In [17]:
categ_df = df.select_dtypes(include=['object', 'category'])
print('кількість колонок категоріального типу:',  categ_df.shape[1])
print('які саме колонки категоріального типу:',  categ_df.columns.tolist())

кількість колонок категоріального типу: 12
які саме колонки категоріального типу: ['continent', 'country', 'device', 'browser', 'mobile_model_name', 'operating_system', 'language', 'traffic_source', 'channel', 'category', 'product_name', 'description']


✅ ***Кількість колонок типу datetime***

In [18]:
datetime_df = df.select_dtypes(include=['dbdate', 'datetime'])
print('кількість колонок числового типу:',  datetime_df.shape[1])
print('які саме колонки числового типу:',  datetime_df.columns.to_list())

кількість колонок числового типу: 1
які саме колонки числового типу: ['date']


✅ ***Кількість унікальних сесій***

In [19]:
print('кількість унікальних сесій:', len(pd.unique(df['session_id'])))

кількість унікальних сесій: 349545


✅ ***Період часу***

In [20]:
print(f'період часу розглядається з {min(df['date'])} до {max(df['date'])}')

період часу розглядається з 2020-11-01 до 2021-01-31


✅ ***Наявність пропущених значень та їх доля***

In [21]:
missing_values = df.isna().sum()
print("Пропущені значення:", missing_values)

missing_percent  = df.isna().mean() * 100
print("\nДоля пропущених значення:", round(missing_percent , 1))

Пропущені значення: date                      0
session_id                0
continent                 0
country                   0
device                    0
browser                   0
mobile_model_name         0
operating_system          0
language             114266
traffic_source            0
channel                   0
account_id           321600
is_verified          321600
is_unsubscribed      321600
category             316007
product_name         316007
price                316007
description          316007
dtype: int64

Доля пропущених значення: date                  0.0
session_id            0.0
continent             0.0
country               0.0
device                0.0
browser               0.0
mobile_model_name     0.0
operating_system      0.0
language             32.7
traffic_source        0.0
channel               0.0
account_id           92.0
is_verified          92.0
is_unsubscribed      92.0
category             90.4
product_name         90.4
price               

In [22]:
print(missing_values[missing_values > 0].sort_values())

language           114266
price              316007
product_name       316007
category           316007
description        316007
is_unsubscribed    321600
account_id         321600
is_verified        321600
dtype: int64


✅ ***Наявність дублікатів***

In [23]:
print("Дублікати:", df.duplicated().sum())

Дублікати: 0


✅ ***Заповнення пропущених значень в колонках:***

 `language` – заповнення значенням ***`Unknown`***;

  `account_id`, `is_verified`, `is_unsubscribed` – гостьові сесії;

  `category`, `product_name`, `description` – сесії без покупки ***`No Purchase`***



---


Заповнення значенням:
  `Unknown`, `No Purchase`. Маніпуляції з даними збереже записи у вибірці, для уникнення викривлення статистики








In [24]:
df['language'] = df['language'].fillna('Unknown')
df['language'].unique()

array(['zh', 'en-us', 'es-es', 'Unknown', 'en-gb', 'en-ca', 'fr', 'ko',
       'en', 'de'], dtype=object)

In [25]:
df['account_id'] = df['account_id'].fillna(0).astype('int64')
df['is_verified'] = df['is_verified'].fillna(0).astype(bool)
df['is_unsubscribed'] = df['is_unsubscribed'].fillna(0).astype(bool)

print(df[['account_id', 'is_verified', 'is_unsubscribed']].isnull().sum())

account_id         0
is_verified        0
is_unsubscribed    0
dtype: int64


In [26]:
df['category'] = df['category'].fillna('No Purchase')
df['product_name'] = df['product_name'].fillna('No Purchase')
df['description'] = df['description'].fillna('No Purchase')

print(df[['category', 'product_name', 'description']].isnull().sum())

category        0
product_name    0
description     0
dtype: int64


In [27]:
df.head()

,date,session_id,continent,country,device,browser,mobile_model_name,operating_system,language,traffic_source,channel,account_id,is_verified,is_unsubscribed,category,product_name,price,description
0,2020-11-01,5760483956,Americas,United States,desktop,Chrome,Safari,Macintosh,zh,<Other>,Paid Search,0,False,False,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
1,2020-11-01,7115337200,Europe,United Kingdom,desktop,Chrome,Chrome,Web,en-us,organic,Organic Search,0,False,False,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
2,2020-11-01,3978035233,Europe,Norway,mobile,Chrome,<Other>,Web,zh,(none),Direct,0,False,False,Tables & desks,RÅSKOG,189.0,"Trolley, 35x45x78 cm"
3,2020-11-01,9648986282,Africa,Nigeria,mobile,Chrome,<Other>,Android,es-es,(none),Direct,0,False,False,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
4,2020-11-01,4393441533,Asia,China,desktop,Chrome,Chrome,Windows,en-us,(none),Direct,0,False,False,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"


Дай відповіді на питання:
на яких континентах (топ-3) та в яких країнах (топ-5) наша компанія має найбільші продажі? найбільшу кількість замовлень?
знайди топ-10 категорій товарів за загальною сумою продажів
знайди топ-10 категорій товарів у країні з найбільшими продажами, чи відрізняється ситуація від загальної?
проаналізуй продажі у розрізі типів та моделей девайсів (у % від загальних продажів)
проаналізуй продажі за джерелами трафіку (у % від загальних продажів)
який відсоток зареєстрованих користувачів підтвердив свою електронну адресу?
який відсоток зареєстрованих користувачів відписався від розсилки?
чи відрізняється поведінка (у плані продажів) тих, хто відписався від розсилки та тих, хто досі підписаний?
в яких країнах найбільше зареєстрованих користувачів?


✅ ***топ-3 континенти та топ-5 країн, в яких компанія має найбільші продажі та найбільшу кількість замовлень***

In [32]:
# копія ДатаСету з покупками, на основі нього подальший аналіз ДатаСету

df_purchases = df[df['price'].notna()].copy()
df_purchases.shape

(33538, 18)

In [47]:
top_continents = df_purchases.groupby('continent').agg(total_profit=('price', 'sum'),count_order=('session_id', 'count')).nlargest(3, 'total_profit')
print("Tоп-3 континенти в яких компанія має найбільші продажі та кількість замовлень")
print(top_continents)

Tоп-3 континенти в яких компанія має найбільші продажі та кількість замовлень
  continent  total_profit  count_order
0  Americas    17665280.0        18553
1      Asia     7601298.3         7950
2    Europe     5934624.2         6261


In [48]:
top_countries = df_purchases.groupby('country').agg(total_profit=('price', 'sum'),count_order=('session_id', 'count')).nlargest(5, 'total_profit').reset_index()
print("Tоп-5 країн в яких компанія має найбільші продажі та кількість замовлень")
print(top_countries)

Tоп-5 країн в яких компанія має найбільші продажі та кількість замовлень
          country  total_profit  count_order
0   United States    13943553.9        14673
1           India     2809762.0         3029
2          Canada     2437921.0         2560
3  United Kingdom      938317.9         1029
4          France      710692.8          678


✅***топ-10 категорій товарів за загальною сумою продажів***

In [50]:
top_categories = df_purchases.groupby('category').agg(total_profit=('price', 'sum')).nlargest(10, 'total_profit').reset_index()
print("Tоп-10 атегорій товарів за загальною сумою продажів")
print(top_categories)

Tоп-10 атегорій товарів за загальною сумою продажів
                           category  total_profit
0                 Sofas & armchairs     8388254.5
1                            Chairs     6147748.8
2                              Beds     4919725.0
3        Bookcases & shelving units     3640818.1
4              Cabinets & cupboards     2336499.5
5                 Outdoor furniture     2142222.2
6                    Tables & desks     1790307.5
7  Chests of drawers & drawer units      906562.5
8                     Bar furniture      735503.0
9              Children's furniture      467697.0


✅ ***топ-10 категорій товарів у країні з найбільшими продажами, чи відрізняється ситуація від загальної?***

In [78]:
# країна з найбільшими продажами

top_country = df_purchases.groupby('country').agg(total_profit=('price', 'sum')).nlargest(1, 'total_profit').index[0]
print("Tоп країна з найбільшими продажами")
print(top_country)

Tоп країна з найбільшими продажами
United States


In [81]:
# топ-10 категорій по цій країні

top_categories_country = (df_purchases[df_purchases['country'] == top_country]
                          .groupby('category').agg(total_profit=('price', 'sum'),count_order=('session_id', 'count'))
                          .nlargest(10, 'total_profit'))
print(top_categories_country)

                                  total_profit  count_order
category                                                   
Sofas & armchairs                    3707144.5         1903
Chairs                               2619773.8         2576
Beds                                 2213058.0         1298
Bookcases & shelving units           1567606.9         3374
Cabinets & cupboards                  994545.5          995
Outdoor furniture                     929245.2          984
Tables & desks                        777865.0         1248
Chests of drawers & drawer units      382388.0          616
Bar furniture                         330805.0          487
Children's furniture                  207575.0          752


In [83]:
# загальна картина для порівняння

top_categories_global = df_purchases.groupby('category').agg(total_profit=('price', 'sum'),count_order=('session_id', 'count')).nlargest(10, 'total_profit')
print(top_categories_global)

                                  total_profit  count_order
category                                                   
Sofas & armchairs                    8388254.5         4301
Chairs                               6147748.8         5952
Beds                                 4919725.0         2926
Bookcases & shelving units           3640818.1         7630
Cabinets & cupboards                 2336499.5         2318
Outdoor furniture                    2142222.2         2229
Tables & desks                       1790307.5         2941
Chests of drawers & drawer units      906562.5         1452
Bar furniture                         735503.0         1092
Children's furniture                  467697.0         1702
